# Classification results

Summarise crop archives and recorded classifier runs. Result dictionaries in this notebook are historical measurements, not training jobs. See FOVEANET_LOG.txt and PREDICTIONS.txt for protocol changes.


## 1. Experimental conditions

Compare equal-shaped input tensors and inspect stored train/test metadata. Earlier archives use different split and crop protocols; keep those results separate.


In [ ]:
import numpy as np

data = np.load("data/crops.npz")
data10 = np.load("data/crops_10ms.npz")
official = np.load("data/crops_official_10ms.npz")
labels, users = data["label"], data["user"]
conditions = ["full", "persistence", "persdens", "centre", "random"]

print(
    f"recordings {len(labels)}, classes {len(np.unique(labels))}, subjects {len(np.unique(users))}"
)
print(
    f"tensor per recording: {data['full'].shape[1:]}  "
    f"({int(np.prod(data['full'].shape[1:]))} numbers, the same in every condition)"
)
sp = official["split"]
print(f"official file: {int((sp == 0).sum())} train + {int((sp == 1).sum())} test recordings")
print(f"  train subjects {sorted(set(official['user'][sp == 0].tolist()))}")
print(f"  test  subjects {sorted(set(official['user'][sp == 1].tolist()))}")
print()
print(f"{'condition':>13}{'50 ms activity':>16}{'10 ms activity':>16}{'empty':>8}")
for c in conditions:
    a, b = data[c], data10[c]
    print(f"{c:>13}{a.mean():>16.2f}{b.mean():>16.2f}{(a.sum(axis=(1, 2, 3, 4)) == 0).sum():>8}")

diff = np.abs(data["full"] - data10["full"])
print(
    f"\nthe 'full' condition crops nothing, yet {100 * (diff > 0).mean():.0f}% of its cells "
    f"differ between windows"
)
print(
    "because events are binned by the frame they fell in, so a different window "
    "moves some between bins"
)

## 2. Event retention

Measure activity retained by each representation. Earlier random-box archives used a different size baseline; use blind_box.py for the corrected same-sized diagnostic.


## 3. Recorded results

The following constants summarise earlier runs. Reproduce an experiment with build_crops.py and classify.py using its recorded window, split, epoch count and seeds.


In [ ]:
results = {
    "improvised 50 ms": {
        "full": (0.701, 0.007, 3),
        "persistence": (0.686, 0.002, 3),
        "persdens": (0.688, 0.007, 3),
        "centre": (0.354, 0.003, 3),
        "random": (0.250, 0.010, 3),
    },
    "improvised 10 ms": {
        "full": (0.7091, 0.0123, 8),
        "persistence": (0.7314, 0.0081, 8),
        "persdens": (0.7082, 0.0110, 8),
        "centre": (0.366, 0.006, 3),
        "random": (0.229, 0.009, 3),
    },
    "OFFICIAL, superseded": {
        "full": (0.855, 0.016, 5),
        "persistence": (0.845, 0.006, 5),
        "persdens": (0.812, 0.010, 5),
        "centre": (0.477, 0.017, 5),
        "random": (0.309, 0.008, 5),
    },
    "OFFICIAL, corrected": {
        "full": (0.875, 0.007, 5),
        "persistence": (0.846, 0.009, 5),
        "persdens": (0.813, 0.015, 5),
        "centre": (0.529, 0.017, 5),
        "random": (0.352, 0.010, 5),
    },
}

final_epoch = {
    "full": 0.861,
    "persistence": 0.837,
    "persdens": 0.805,
    "centre": 0.511,
    "random": 0.327,
}

sweep = {
    "fov03": (0.030, 0.779),
    "fov10": (0.099, 0.817),
    "fov25": (0.237, 0.869),
    "fov50": (0.436, 0.889),
    "fov75": (0.595, 0.868),
    "full": (1.000, 0.861),
    "cen03": (0.030, 0.472),
    "cen10": (0.100, 0.655),
    "cen25": (0.250, 0.746),
    "cen50": (0.500, 0.843),
    "cen75": (0.750, 0.849),
}

diagnostic = {
    "DVS128 Gesture": (0.192, +0.326),
    "SL-Animals": (0.550, +0.059),
    "UCF-50": (0.592, -0.063),
    "DVS-Lip": (0.627, -0.143),
}

print("produced by classify.py on data/crops_official_10ms_v2.npz and")
print("data/crops_sweep_10ms.npz, 5 seeds, train on subjects 1-23, test on 24-29\n")

print(f"{'condition':>13}{'superseded':>13}{'corrected':>12}{'final epoch':>14}")
for c in conditions:
    print(
        f"{c:>13}{results['OFFICIAL, superseded'][c][0]:>13.3f}"
        f"{results['OFFICIAL, corrected'][c][0]:>12.3f}{final_epoch[c]:>14.3f}"
    )
print("\nthe superseded column used controls sized from the persdens boxes, about a")
print("fifth too small in area, and an epoch chosen on the test set.\n")


def sigma(table, a, b):
    ma, sa, na = table[a]
    mb, sb, nb = table[b]
    se = np.sqrt(sa**2 / na + sb**2 / nb)
    return ma - mb, abs(ma - mb) / se


print("the comparisons that matter, on the corrected run")
for a, b in [
    ("persistence", "centre"),
    ("persistence", "full"),
    ("persistence", "persdens"),
    ("centre", "random"),
]:
    d, z = sigma(results["OFFICIAL, corrected"], a, b)
    print(
        f"  {a:>11} - {b:<11} {d:>+8.4f}   {z:>5.1f} sigma   "
        f"{'real' if z > 2.5 else 'not significant'}"
    )

print("\nkept fraction against accuracy, placed two ways")
print(f"{'kept':>8}{'fovea':>9}{'centre':>9}{'gap':>9}")
for f, c in (
    ("fov03", "cen03"),
    ("fov10", "cen10"),
    ("fov25", "cen25"),
    ("fov50", "cen50"),
    ("fov75", "cen75"),
):
    kept, fa = sweep[f]
    _, ca = sweep[c]
    print(f"{100 * kept:>7.1f}%{fa:>9.3f}{ca:>9.3f}{fa - ca:>+9.3f}")
print(f"{100.0:>7.1f}%{sweep['full'][1]:>9.3f}{sweep['full'][1]:>9.3f}{0.0:>+9.3f}")

print("\nblind-box ratio against the placement premium, four datasets")
for name, (ratio, premium) in diagnostic.items():
    print(f"  {name:>16}  ratio {ratio:.3f}   premium {premium:+.3f}")

## Interpreting the table

Compare conditions within the same protocol. Validation-selected, final-epoch and historical maximum-test results are not interchangeable.


## 4. UCF-50 comparison

Inspect UCF-50 results separately from gesture results. Its sensor dimensions and recording conditions differ. Values below are recorded summaries.


In [ ]:
ucf = np.load("data/crops_ucf.npz")
ucf_results = {
    "full": (0.698, 0.020),
    "centre": (0.589, 0.026),
    "persdens": (0.543, 0.014),
    "persistence": (0.526, 0.017),
    "random": (0.450, 0.018),
}
n_seeds = 5

print(
    f"UCF-50: {len(ucf['label'])} recordings, {len(np.unique(ucf['label']))} classes, "
    f"{len(np.unique(ucf['user']))} groups"
)
print(f"held out 6 groups: 351 train, 129 test.  chance {1 / 12:.3f}\n")

print(f"{'condition':>13}{'DVSGesture':>12}{'UCF-50':>9}")
gesture_10ms = {
    "full": 0.709,
    "persistence": 0.731,
    "persdens": 0.708,
    "centre": 0.366,
    "random": 0.229,
}
for c in conditions:
    print(f"{c:>13}{gesture_10ms[c]:>12.3f}{ucf_results[c][0]:>9.3f}")

print("\nUCF-50 orderings, in standard errors of the difference")
for a, b in [
    ("full", "centre"),
    ("centre", "persistence"),
    ("persistence", "random"),
    ("persdens", "persistence"),
]:
    ma, sa = ucf_results[a]
    mb, sb = ucf_results[b]
    se = np.sqrt(sa**2 / n_seeds + sb**2 / n_seeds)
    print(f"  {a:>11} - {b:<11} {ma - mb:>+7.3f}   {abs(ma - mb) / se:>4.1f} se")

print("\nblind-box ratio, from blind_box.py over 120 recordings per dataset")
print(f"{'dataset':>16}{'ratio':>8}{'premium':>10}")
for name, (ratio, premium) in diagnostic.items():
    print(f"{name:>16}{ratio:>8.3f}{premium:>+10.3f}")
print("\nNOT computed from these archives. An earlier version of this cell divided the")
print("centre condition's activity by the fovea's and reported 0.11 and 0.47. That is")
print("the wrong quantity twice over: the centre box is a deliberate prior rather than")
print("a blind placement, and the controls in those archives were sized from the")
print("persdens boxes and so about a fifth too small. blind_box.py places the box at")
print("random and sizes it from the rule it is compared against.")

## 5. Generalisation

A compact active region can help crop placement, but activity alone does not identify the task target. Consult the prediction record for failures and recording-condition dependence.
